In [119]:
import pandas as pd
import os
import re
import numpy as np
from pathlib import Path

In [120]:
SSP_MODELING = Path.cwd().resolve().parents[1]
TABLEAU_DATA = SSP_MODELING / "tableau" / "data"
SCRIPT_DIR_PATH = os.getcwd()
CB_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
SSP_MODELING_DIR_PATH = os.path.dirname(CB_DIR_PATH)
TORNADO_DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")
INPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "input")
OUTPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "output")


In [121]:
TABLEAU_DATA

PosixPath('/Users/fabianfuentes/git/ssp_libya/ssp_modeling/tableau/data')

In [122]:
def add_sector_and_transformation_fields(df: pd.DataFrame, strategy_col: str = "strategy") -> pd.DataFrame:
    """
    Creates:
      - sector: sector code (e.g., AGRC)
      - transformation_name: transformation text after '... - <SECTOR>:'
    """
    df = df.copy()

    # --- sector extraction (captures 3-6 uppercase letters before colon) ---
    # Example: "Singleton - Default Value - AGRC: Improve rice..." -> AGRC
    df["sector"] = df[strategy_col].str.extract(r"-\s*([A-Z]{3,6})\s*:", expand=False)

    # Special case: baseline strategy
    df.loc[df[strategy_col].str.contains(r"^Strategy\s+TX:BASE", regex=True, na=False), "sector"] = "BASE"

    # --- transformation_name extraction ---
    # Keep only text after "<SECTOR>:"
    # Example -> "Improve rice..."
    df["transformation_name"] = df[strategy_col].str.extract(r":\s*(.*)$", expand=False)

    # If it's baseline, keep the full strategy string as the name (or label it as BASE)
    base_mask = df[strategy_col].str.contains(r"^Strategy\s+TX:BASE", regex=True, na=False)
    df.loc[base_mask, "transformation_name"] = "BASE"

    # Clean whitespace
    df["transformation_name"] = df["transformation_name"].fillna("").str.strip()

    return df

## Load and process emission data

In [123]:
# Load the decomposed emissions long format data
emissions_df = pd.read_csv(os.path.join(TABLEAU_DATA, "decomposed_emissions_libya_2023.csv"))
emissions_df.head()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
0,0.0,0.0,1.A.1 - Electricity and Heat Generation:CH4,1 - Energy,1.A.1 - Electricity and Heat Generation,0.025473,2023,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE,0.025473,NaN
1,0.0,0.0,1.A.1 - Electricity and Heat Generation:CH4,1 - Energy,1.A.1 - Electricity and Heat Generation,0.010848,2024,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE,0.010848,NaN
2,0.0,0.0,1.A.1 - Electricity and Heat Generation:CH4,1 - Energy,1.A.1 - Electricity and Heat Generation,0.011016,2025,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE,0.011016,NaN
3,0.0,0.0,1.A.1 - Electricity and Heat Generation:CH4,1 - Energy,1.A.1 - Electricity and Heat Generation,0.011187,2026,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE,0.011187,NaN
4,0.0,0.0,1.A.1 - Electricity and Heat Generation:CH4,1 - Energy,1.A.1 - Electricity and Heat Generation,0.011358,2027,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE,0.011358,NaN


In [124]:
print(emissions_df.primary_id.nunique())

4


In [125]:
print(emissions_df.primary_id.nunique())

4


In [126]:
# check unique strategy
emissions_df['strategy'].unique()

array(['Strategy TX:BASE', 'BAU', 'Unconditional', 'Conditional',
       'Historical'], dtype=object)

In [127]:
# Drop historical and tx:base from df
filtered_emissions_df = emissions_df.loc[~emissions_df['strategy'].isin(['Historical', 'Strategy TX:BASE'])]
print(emissions_df['strategy'].nunique())
print(filtered_emissions_df['strategy'].nunique())

5
3


In [128]:
filtered_emissions_df["strategy"].unique()

array(['BAU', 'Unconditional', 'Conditional'], dtype=object)

In [129]:
filtered_emissions_df.tail()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
5035,6005.0,76076.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2046,N2O,0.0,0.0,Conditional,LBY,libya,SISEPUEDE,0.0,NaN
5036,6005.0,76076.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2047,N2O,0.0,0.0,Conditional,LBY,libya,SISEPUEDE,0.0,NaN
5037,6005.0,76076.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2048,N2O,0.0,0.0,Conditional,LBY,libya,SISEPUEDE,0.0,NaN
5038,6005.0,76076.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2049,N2O,0.0,0.0,Conditional,LBY,libya,SISEPUEDE,0.0,NaN
5039,6005.0,76076.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2050,N2O,0.0,0.0,Conditional,LBY,libya,SISEPUEDE,0.0,NaN


In [130]:
# Now concat the original base df and the filtered emissions df
tornado_emissions_df = filtered_emissions_df
tornado_emissions_df['strategy'].unique()

array(['BAU', 'Unconditional', 'Conditional'], dtype=object)

In [131]:
tornado_emissions_df.head()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
1260,6003.0,74074.0,1.A.1 - Electricity and Heat Generation:CH4,1 - Energy,1.A.1 - Electricity and Heat Generation,0.025473,2023,CH4,0.0,0.0,BAU,LBY,libya,SISEPUEDE,0.025473,NaN
1261,6003.0,74074.0,1.A.1 - Electricity and Heat Generation:CH4,1 - Energy,1.A.1 - Electricity and Heat Generation,0.010848,2024,CH4,0.0,0.0,BAU,LBY,libya,SISEPUEDE,0.010848,NaN
1262,6003.0,74074.0,1.A.1 - Electricity and Heat Generation:CH4,1 - Energy,1.A.1 - Electricity and Heat Generation,0.011016,2025,CH4,0.0,0.0,BAU,LBY,libya,SISEPUEDE,0.011016,NaN
1263,6003.0,74074.0,1.A.1 - Electricity and Heat Generation:CH4,1 - Energy,1.A.1 - Electricity and Heat Generation,0.011187,2026,CH4,0.0,0.0,BAU,LBY,libya,SISEPUEDE,0.011187,NaN
1264,6003.0,74074.0,1.A.1 - Electricity and Heat Generation:CH4,1 - Energy,1.A.1 - Electricity and Heat Generation,0.011358,2027,CH4,0.0,0.0,BAU,LBY,libya,SISEPUEDE,0.011358,NaN


In [132]:
tornado_emissions_df.loc[
    tornado_emissions_df["subsector"].str.startswith("2."), "subsector"
] = "2 - IPPU"
tornado_emissions_df.loc[
    tornado_emissions_df["subsector"].str.startswith("1.A.1"), "subsector"
] = "1.A.1 - Energy Industries"

In [133]:
# Aggregate by strategy_id, primary_id and strategy, and sum value
tornado_emissions_agg_df = tornado_emissions_df.groupby(
    ['strategy_id', 'primary_id', 'strategy', 'sector', 'subsector']
)['value'].sum().reset_index()

tornado_emissions_agg_df.head()


,strategy_id,primary_id,strategy,sector,subsector,value
0,6003.0,74074.0,BAU,1 - Energy,1.A.1 - Energy Industries,1113.854784
1,6003.0,74074.0,BAU,1 - Energy,1.A.2 - Manufacturing Industries and Construction,58.966287
2,6003.0,74074.0,BAU,1 - Energy,1.A.3 - Transport,676.890490
3,6003.0,74074.0,BAU,1 - Energy,1.A.4 - Other Sectors,25.312214
4,6003.0,74074.0,BAU,1 - Energy,1.B - Fugitive emissions from fuels,793.469912


In [134]:
tornado_emissions_agg_df.tail()

,strategy_id,primary_id,strategy,sector,subsector,value
31,6005.0,76076.0,Conditional,"3 - Agriculture, Forestry, and Other Land Use",3.B - Land,-31.936842
32,6005.0,76076.0,Conditional,"3 - Agriculture, Forestry, and Other Land Use",3.C - Aggregate sources and non-CO2 emissions ...,23.014057
33,6005.0,76076.0,Conditional,4 - Waste,4.A - Solid Waste Disposal,14.656217
34,6005.0,76076.0,Conditional,4 - Waste,4.D - Wastewater Treatment and Discharge,13.686307
35,6005.0,76076.0,Conditional,5- CCSQ,5- CCSQ,0.000000


In [135]:
tornado_emissions_agg_df['subsector'].unique()

array(['1.A.1 - Energy Industries',
       '1.A.2 - Manufacturing Industries and Construction',
       '1.A.3 - Transport', '1.A.4 - Other Sectors',
       '1.B - Fugitive emissions from fuels', '2 - IPPU',
       '3.A - Livestock', '3.B - Land',
       '3.C - Aggregate sources and non-CO2 emissions sources on land',
       '4.A - Solid Waste Disposal',
       '4.D - Wastewater Treatment and Discharge', '5- CCSQ'],
      dtype=object)

In [136]:
# check if strategy id nunique matches amount of rows
print(tornado_emissions_agg_df['strategy_id'].nunique())
print(tornado_emissions_agg_df.shape[0])

3
36


In [137]:
# rename value to emission_total
tornado_emissions_agg_df = tornado_emissions_agg_df.rename(columns={'value': 'emission_total'})

# create base_emission_total per subsector using strategy_id == 6003
base_by_subsector = tornado_emissions_agg_df.loc[
    tornado_emissions_agg_df['strategy_id'] == 6003, ['subsector', 'emission_total']
].rename(columns={'emission_total': 'base_emission_total'})

tornado_emissions_agg_df = tornado_emissions_agg_df.merge(base_by_subsector, on='subsector', how='left')

# calculate emission difference column
tornado_emissions_agg_df['emission_diff'] = tornado_emissions_agg_df['emission_total'] - tornado_emissions_agg_df['base_emission_total']

tornado_emissions_agg_df['emission_diff'] = tornado_emissions_agg_df['emission_diff'].round(1)

tornado_emissions_agg_df.head(40)

,strategy_id,primary_id,strategy,sector,subsector,emission_total,base_emission_total,emission_diff
0,6003.0,74074.0,BAU,1 - Energy,1.A.1 - Energy Industries,1113.854784,1113.854784,0.0
1,6003.0,74074.0,BAU,1 - Energy,1.A.2 - Manufacturing Industries and Construction,58.966287,58.966287,0.0
2,6003.0,74074.0,BAU,1 - Energy,1.A.3 - Transport,676.890490,676.890490,0.0
3,6003.0,74074.0,BAU,1 - Energy,1.A.4 - Other Sectors,25.312214,25.312214,0.0
4,6003.0,74074.0,BAU,1 - Energy,1.B - Fugitive emissions from fuels,793.469912,793.469912,0.0
5,6003.0,74074.0,BAU,2 - Industrial Processes and Product Use,2 - IPPU,209.195632,209.195632,0.0
6,6003.0,74074.0,BAU,"3 - Agriculture, Forestry, and Other Land Use",3.A - Livestock,32.946742,32.946742,0.0
7,6003.0,74074.0,BAU,"3 - Agriculture, Forestry, and Other Land Use",3.B - Land,-28.554219,-28.554219,0.0
8,6003.0,74074.0,BAU,"3 - Agriculture, Forestry, and Other Land Use",3.C - Aggregate sources and non-CO2 emissions ...,20.082066,20.082066,0.0
9,6003.0,74074.0,BAU,4 - Waste,4.A - Solid Waste Disposal,24.449927,24.449927,0.0


In [138]:
tornado_emissions_agg_df['strategy'].unique()

array(['BAU', 'Unconditional', 'Conditional'], dtype=object)

In [139]:
tornado_emissions_agg_df = tornado_emissions_agg_df[tornado_emissions_agg_df['strategy']!='NDC'] 

In [140]:
tornado_emissions_agg_df['strategy'].unique()

array(['BAU', 'Unconditional', 'Conditional'], dtype=object)

## Load and process CB data

In [141]:
# cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "costs_benefits_sisepuede_results_sisepuede_run_2026-01-29T15;28;40.322709_tornado_raw.csv"))
cb_raw_df = pd.read_csv(os.path.join(TABLEAU_DATA, "cb_data.csv"))
cb_raw_df.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,...,sector,cb_type,item_1,item_2,Year,strategy,strategy_id,primary_id,ids,gdp_mmm_usd
0,PFLO:BAU,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2023.0,Business as Usual,6003,74074,cb:agrc:crop_value:crops_produced:bevs_and_spi...,101.162039
1,PFLO:BAU,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2024.0,Business as Usual,6003,74074,cb:agrc:crop_value:crops_produced:bevs_and_spi...,105.576742
2,PFLO:BAU,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2025.0,Business as Usual,6003,74074,cb:agrc:crop_value:crops_produced:bevs_and_spi...,107.688276
3,PFLO:BAU,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2026.0,Business as Usual,6003,74074,cb:agrc:crop_value:crops_produced:bevs_and_spi...,109.842042
4,PFLO:BAU,0.0,libya,12.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2027.0,Business as Usual,6003,74074,cb:agrc:crop_value:crops_produced:bevs_and_spi...,112.038883


In [142]:
# --- Create a copy of the raw data ---
cb_data = cb_raw_df.copy()

cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,...,sector,cb_type,item_1,item_2,Year,strategy,strategy_id,primary_id,ids,gdp_mmm_usd
0,PFLO:BAU,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2023.0,Business as Usual,6003,74074,cb:agrc:crop_value:crops_produced:bevs_and_spi...,101.162039
1,PFLO:BAU,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2024.0,Business as Usual,6003,74074,cb:agrc:crop_value:crops_produced:bevs_and_spi...,105.576742
2,PFLO:BAU,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2025.0,Business as Usual,6003,74074,cb:agrc:crop_value:crops_produced:bevs_and_spi...,107.688276
3,PFLO:BAU,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2026.0,Business as Usual,6003,74074,cb:agrc:crop_value:crops_produced:bevs_and_spi...,109.842042
4,PFLO:BAU,0.0,libya,12.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2027.0,Business as Usual,6003,74074,cb:agrc:crop_value:crops_produced:bevs_and_spi...,112.038883


In [143]:
cb_data["sector"].unique()

array(['agrc', 'ccsq', 'enfu', 'entc', 'inen', 'ippu', 'lndu', 'lsmm',
       'lvst', 'scoe', 'soil', 'trns', 'trww', 'wali', 'waso', 'fgtv'],
      dtype=object)

In [144]:
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,...,sector,cb_type,item_1,item_2,Year,strategy,strategy_id,primary_id,ids,gdp_mmm_usd
0,PFLO:BAU,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2023.0,Business as Usual,6003,74074,cb:agrc:crop_value:crops_produced:bevs_and_spi...,101.162039
1,PFLO:BAU,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2024.0,Business as Usual,6003,74074,cb:agrc:crop_value:crops_produced:bevs_and_spi...,105.576742
2,PFLO:BAU,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2025.0,Business as Usual,6003,74074,cb:agrc:crop_value:crops_produced:bevs_and_spi...,107.688276
3,PFLO:BAU,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2026.0,Business as Usual,6003,74074,cb:agrc:crop_value:crops_produced:bevs_and_spi...,109.842042
4,PFLO:BAU,0.0,libya,12.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2027.0,Business as Usual,6003,74074,cb:agrc:crop_value:crops_produced:bevs_and_spi...,112.038883


In [145]:
cb_data['cb_type'].unique()

array(['crop_value', 'sector_specific', 'fuel_cost', 'air_pollution',
       'technical_cost', 'ippu_value', 'ecosystem_services', 'lvst_value',
       'land_pollution', 'congestion', 'road_safety', 'system_cost',
       'water_pollution', 'human_health', 'env_pollution',
       'consumer_savings', 'technical_savings'], dtype=object)

In [146]:
cb_data = cb_data[cb_data['cb_type']=='technical_cost']

In [147]:
cb_data['cb_type'].unique()

array(['technical_cost'], dtype=object)

In [148]:
cb_data['strategy_code'].unique()

array(['PFLO:BAU', 'PFLO:UNCONDITIONAL', 'PFLO:CONDITIONAL'], dtype=object)

In [149]:
cb_data['strategy_code'].unique()

array(['PFLO:BAU', 'PFLO:UNCONDITIONAL', 'PFLO:CONDITIONAL'], dtype=object)

In [150]:
cb_data = cb_data[cb_data['strategy_code']!='BASE']

In [151]:
cb_data['strategy_code'].unique()

array(['PFLO:BAU', 'PFLO:UNCONDITIONAL', 'PFLO:CONDITIONAL'], dtype=object)

In [152]:
tornado_emissions_agg_df['subsector'].unique()

array(['1.A.1 - Energy Industries',
       '1.A.2 - Manufacturing Industries and Construction',
       '1.A.3 - Transport', '1.A.4 - Other Sectors',
       '1.B - Fugitive emissions from fuels', '2 - IPPU',
       '3.A - Livestock', '3.B - Land',
       '3.C - Aggregate sources and non-CO2 emissions sources on land',
       '4.A - Solid Waste Disposal',
       '4.D - Wastewater Treatment and Discharge', '5- CCSQ'],
      dtype=object)

In [153]:
SECTOR_TO_SUBSECTOR = {
    
    "entc": "1.A.1 - Energy Industries",
    "inen": "1.A.2 - Manufacturing Industries and Construction",
    "trns": "1.A.3 - Transport",
    "scoe": "1.A.4 - Other Sectors",

    'ippu': '2 - IPPU',

    "fgtv": "1.B - Fugitive emissions from fuels",

    "lvst": "3.A - Livestock",
    "lsmm": "3.A - Livestock",

    "soil": "3.C - Aggregate sources and non-CO2 emissions sources on land",
    "agrc": "3.C - Aggregate sources and non-CO2 emissions sources on land",
    "agr": "3.C - Aggregate sources and non-CO2 emissions sources on land",

    "lndu": "3.B - Land",

    "waso": "4.A - Solid Waste Disposal",
    "trww": "4.D - Wastewater Treatment and Discharge",

    "ccsq": "5- CCSQ",

}

def add_subsector(df: pd.DataFrame, sector_col: str = "sector") -> pd.DataFrame:
    df = df.copy()
    df["subsector_ssp"] = df[sector_col].str.lower().map(SECTOR_TO_SUBSECTOR)
    return df

cb_data = add_subsector(cb_data)

In [154]:
# aggregate sum(value) grouped by strategy_code and sector
cb_data = (
    cb_data.groupby(["strategy_code", "strategy_id", "primary_id", "subsector_ssp"], as_index=False)["value"]
      .sum()
      .rename(columns={"value": "cumulative"})
)
cb_data.head(20)

,strategy_code,strategy_id,primary_id,subsector_ssp,cumulative
0,PFLO:BAU,6003,74074,1.A.1 - Energy Industries,-4.727611e-04
1,PFLO:BAU,6003,74074,3.A - Livestock,-5.294981e-02
2,PFLO:BAU,6003,74074,3.C - Aggregate sources and non-CO2 emissions ...,0.000000e+00
3,PFLO:BAU,6003,74074,4.A - Solid Waste Disposal,-1.165178e-01
4,PFLO:BAU,6003,74074,4.D - Wastewater Treatment and Discharge,-1.356040e+01
5,PFLO:CONDITIONAL,6005,76076,1.A.1 - Energy Industries,-3.569192e+01
6,PFLO:CONDITIONAL,6005,76076,1.A.2 - Manufacturing Industries and Construction,-6.008038e+00
7,PFLO:CONDITIONAL,6005,76076,1.A.3 - Transport,-4.724264e+00
8,PFLO:CONDITIONAL,6005,76076,1.A.4 - Other Sectors,-1.560918e+00
9,PFLO:CONDITIONAL,6005,76076,1.B - Fugitive emissions from fuels,-5.440386e+00


In [155]:
tornado_emissions_agg_df.head()

,strategy_id,primary_id,strategy,sector,subsector,emission_total,base_emission_total,emission_diff
0,6003.0,74074.0,BAU,1 - Energy,1.A.1 - Energy Industries,1113.854784,1113.854784,0.0
1,6003.0,74074.0,BAU,1 - Energy,1.A.2 - Manufacturing Industries and Construction,58.966287,58.966287,0.0
2,6003.0,74074.0,BAU,1 - Energy,1.A.3 - Transport,676.890490,676.890490,0.0
3,6003.0,74074.0,BAU,1 - Energy,1.A.4 - Other Sectors,25.312214,25.312214,0.0
4,6003.0,74074.0,BAU,1 - Energy,1.B - Fugitive emissions from fuels,793.469912,793.469912,0.0


In [156]:
merged_df = tornado_emissions_agg_df.merge(
    cb_data[["strategy_id", "primary_id", "subsector_ssp", "cumulative"]],
    left_on=["strategy_id", "primary_id", "subsector"],
    right_on=["strategy_id", "primary_id", "subsector_ssp"],
    how="left"
).drop(columns="subsector_ssp")

merged_df.tail(20)
  

,strategy_id,primary_id,strategy,sector,subsector,emission_total,base_emission_total,emission_diff,cumulative
16,6004.0,75075.0,Unconditional,1 - Energy,1.B - Fugitive emissions from fuels,651.940051,793.469912,-141.5,-3.429860e+00
17,6004.0,75075.0,Unconditional,2 - Industrial Processes and Product Use,2 - IPPU,164.789835,209.195632,-44.4,-6.241596e-08
18,6004.0,75075.0,Unconditional,"3 - Agriculture, Forestry, and Other Land Use",3.A - Livestock,32.938624,32.946742,-0.0,-5.287742e-02
19,6004.0,75075.0,Unconditional,"3 - Agriculture, Forestry, and Other Land Use",3.B - Land,-31.936842,-28.554219,-3.4,-1.523585e-01
20,6004.0,75075.0,Unconditional,"3 - Agriculture, Forestry, and Other Land Use",3.C - Aggregate sources and non-CO2 emissions ...,21.851610,20.082066,1.8,1.796496e-03
21,6004.0,75075.0,Unconditional,4 - Waste,4.A - Solid Waste Disposal,24.431177,24.449927,-0.0,-1.076471e-01
22,6004.0,75075.0,Unconditional,4 - Waste,4.D - Wastewater Treatment and Discharge,13.686307,13.686307,0.0,-1.356040e+01
23,6004.0,75075.0,Unconditional,5- CCSQ,5- CCSQ,0.000000,0.000000,0.0,NaN
24,6005.0,76076.0,Conditional,1 - Energy,1.A.1 - Energy Industries,743.071731,1113.854784,-370.8,-3.569192e+01
25,6005.0,76076.0,Conditional,1 - Energy,1.A.2 - Manufacturing Industries and Construction,32.436276,58.966287,-26.5,-6.008038e+00


In [157]:
OUTPUT_DATA_DIR_PATH

'/Users/fabianfuentes/git/ssp_libya/ssp_modeling/cost-benefits/tornado_plot/data/output'

In [158]:
merged_df.to_csv(os.path.join(TABLEAU_DATA, "mac_sector_libya.csv"), index=False)